In [ ]:
import pandas as pd
import sksurv
from sksurv.ensemble import RandomSurvivalForest
from sksurv.util import Surv
from sksurv.preprocessing import OneHotEncoder


In [ ]:
RSF_df = pd.read_csv("RSF_data_clean.csv")

y = Surv.from_dataframe("event", "time", RSF_df)

X = RSF_df.drop(columns=['event', 'time'])

drop_cols = [
    'Unnamed: 0',
    'LAT_016', 'LONG_017', 'KILOPOINT_011',
    # bridge_age == time by construction (time is derived from it), so leaving it
    # in X leaks the target and inflates the C-index. The maintenance flags are
    # now dropped at encoding in Build_RSF (era leakage).
    'bridge_age',
] + [c for c in X.columns if c.startswith('COUNTY_')]

X = X.drop(columns=drop_cols)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [ ]:
from sksurv.ensemble import RandomSurvivalForest

rsf = RandomSurvivalForest(
    n_estimators=1000,
    min_samples_split=10,
    min_samples_leaf=5,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

rsf.fit(X_train, y_train)

In [ ]:
c_index = rsf.score(X_test, y_test)

print(f"C-index: {c_index:.3f}")

In [ ]:
df = pd.read_csv('RSF_data_a.csv')

survival_functions = rsf.predict_survival_function(X)

bridge_ids = df[['STRUCTURE_NUMBER_008', 'YEAR_BUILT_027', 'COUNTY_CODE_003', 'PLACE_CODE_004', 'LAT_016', 'LONG_017']].copy()

results = []
for i, sf in enumerate(survival_functions):
    times = sf.x
    probs = sf(times)
    
    below_threshold = times[probs < 0.5]
    predicted_time = below_threshold[0] if len(below_threshold) > 0 else None
    
    idx = X.index[i]
    year_built = bridge_ids.loc[idx, 'YEAR_BUILT_027']
    
    predicted_calendar_year = (
        int(year_built + predicted_time)
        if predicted_time is not None else None
    )
    
    years_remaining = (
        int(predicted_calendar_year - 2025)
        if predicted_calendar_year is not None else None
    )
    
    results.append({
        'STRUCTURE_NUMBER_008': bridge_ids.loc[idx, 'STRUCTURE_NUMBER_008'],
        'YEAR_BUILT_027': year_built,
        'COUNTY_CODE_003': bridge_ids.loc[idx, 'COUNTY_CODE_003'],
        'PLACE_CODE_004': bridge_ids.loc[idx, 'PLACE_CODE_004'],
        'LAT_016': bridge_ids.loc[idx, 'LAT_016'],
        'LONG_017': bridge_ids.loc[idx, 'LONG_017'],
        'predicted_age_at_poor': predicted_time,
        'predicted_year_goes_poor': predicted_calendar_year,
        'years_remaining': years_remaining,
        'risk_category': 'High Risk' if predicted_calendar_year is not None else 'Low Risk'
    })

results_df = pd.DataFrame(results)
results_df.to_csv("RESULTS_years_until_failure.csv")

In [ ]:
bridge_ids = df[['STRUCTURE_NUMBER_008', 'YEAR_BUILT_027', 'COUNTY_CODE_003', 'PLACE_CODE_004', 'LAT_016', 'LONG_017']].copy()

results = []
for i, sf in enumerate(survival_functions):
    times = sf.x
    probs = sf(times)
    
    idx = X.index[i]
    year_built = bridge_ids.loc[idx, 'YEAR_BUILT_027']
    current_age = 2025 - year_built
    
    domain_max = sf.domain[1]
    
    # Clip so we never query outside the domain
    age_5 = min(current_age + 5, domain_max)
    age_10 = min(current_age + 10, domain_max)
    age_20 = min(current_age + 20, domain_max)
    
    prob_fail_5 = 1 - sf(age_5)
    prob_fail_10 = 1 - sf(age_10)
    prob_fail_20 = 1 - sf(age_20)
    
    below_threshold = times[probs < 0.5]
    predicted_time = below_threshold[0] if len(below_threshold) > 0 else None
    
    predicted_calendar_year = (
        int(year_built + predicted_time)
        if predicted_time is not None else None
    )
    
    years_remaining = (
        int(predicted_calendar_year - 2025)
        if predicted_calendar_year is not None else None
    )
    
    results.append({
        'STRUCTURE_NUMBER_008': bridge_ids.loc[idx, 'STRUCTURE_NUMBER_008'],
        'YEAR_BUILT_027': year_built,
        'COUNTY_CODE_003': bridge_ids.loc[idx, 'COUNTY_CODE_003'],
        'PLACE_CODE_004': bridge_ids.loc[idx, 'PLACE_CODE_004'],
        'LAT_016': bridge_ids.loc[idx, 'LAT_016'],
        'LONG_017': bridge_ids.loc[idx, 'LONG_017'],
        'current_age': current_age,
        'prob_fail_5yr': round(prob_fail_5, 3),
        'prob_fail_10yr': round(prob_fail_10, 3),
        'prob_fail_20yr': round(prob_fail_20, 3),
        'predicted_age_at_poor': predicted_time,
        'predicted_year_goes_poor': predicted_calendar_year,
        'years_remaining': years_remaining,
        'risk_category': 'High Risk' if predicted_calendar_year is not None else 'Low Risk'
    })

results_df = pd.DataFrame(results)
results_df.to_csv("RESULTS_prob_of_failure.csv")
print(results_df.sort_values('prob_fail_10yr', ascending=False).head(20))

In [ ]:
import json

cols = ['STRUCTURE_NUMBER_008', 'LAT_016', 'LONG_017', 
        'prob_fail_5yr', 'prob_fail_10yr', 'prob_fail_20yr']

print(results_df[cols].to_json(orient='records'))

In [ ]:
#Tuning

# ── Stage 1: Randomized search with low n_estimators ──────────────────
from sklearn.model_selection import RandomizedSearchCV
from sksurv.ensemble import RandomSurvivalForest
from scipy.stats import randint
import joblib

param_dist = {
    "min_samples_split": randint(5, 30),
    "min_samples_leaf":  randint(3, 20),
    "max_features":      ["sqrt", 0.2, 0.3, 0.4],
    "max_depth":         [None, 10, 15, 20],
}

rsf_fast = RandomSurvivalForest(
    n_estimators=100,   # low for speed during search
    n_jobs=-1,
    random_state=42
)

random_search = RandomizedSearchCV(
    rsf_fast,
    param_distributions=param_dist,
    n_iter=50,          # 50 combinations × 5 folds = 250 fits
    cv=5,
    n_jobs=-1,
    random_state=42,
    verbose=2
)

random_search.fit(X_train, y_train)

print("Best params:", random_search.best_params_)
print("Search C-index (100 trees):", round(random_search.best_score_, 3))

# Save search results in case you need them later
joblib.dump(random_search, "random_search_results.pkl")


# ── Stage 2: Refit best params with full n_estimators ─────────────────
final_rsf = RandomSurvivalForest(
    n_estimators=500,
    **random_search.best_params_,
    n_jobs=-1,
    random_state=42
)

final_rsf.fit(X_train, y_train)

c_index_final = final_rsf.score(X_test, y_test)
print(f"Final C-index (500 trees): {c_index_final:.3f}")

# Save the final model
joblib.dump(final_rsf, "rsf_final_model.pkl")


In [ ]:
print(results_df.head())